
---



1. Qu’est-ce que le NLI ?

NLI (Natural Language Inference) est une tâche où le modèle reçoit deux phrases :

la prémisse : une phrase de départ ;
l’hypothèse : une phrase à évaluer.

Il doit répondre : la prémisse implique l’hypothèse (vrai), la contredit (faux), ou n’a pas de lien (neutre).

2. Le modèle n’a jamais vu de label « sarcasme » pendant son entraînement. Il a seulement appris la relation d’implication entre phrases, sur des données NLI générales.

3. PR-AUC : mesure la qualité du classement, indépendante du seuil

In [2]:
import os, sys
REPO = "/content/tp2-sarcasme-arabe"
if not os.path.exists(REPO):
    !git clone https://github.com/AbdelhamidRb/tp2-sarcasme-arabe.git
os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
!git pull

remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 1.94 KiB | 661.00 KiB/s, done.
From https://github.com/AbdelhamidRb/tp2-sarcasme-arabe
   3978071..4f5cc42  main       -> origin/main
Updating 3978071..4f5cc42
Fast-forward
 notebooks/02_partie_A_baseline.ipynb | 196 ++++++++++++++++++++---------------
 results/results.csv                  |   2 +
 2 files changed, 115 insertions(+), 83 deletions(-)


In [3]:
from src.seed import set_seed
from src.data import load_splits
from src.preprocess import dedupliquer_train

set_seed(42)
train_brut, val, test = load_splits()
train = dedupliquer_train(train_brut, test)
print("val :", len(val), "| test :", len(test))

train avant : 11293
  fuite retirée (présents dans test) : 4
  groupes contradictoires retirés    : 13
train après : 11217
val : 1255 | test : 3000


In [4]:
from transformers import pipeline

ZS_MODEL = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
zs = pipeline("zero-shot-classification", model=ZS_MODEL, device=0)

exemple = val.tweet.iloc[0]
print(zs(exemple,
         candidate_labels=["ساخرة", "جادة"],
         hypothesis_template="هذه التغريدة {}."))

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

{'sequence': '"بشار الجعفري يشن هجوما حادا على السفيرة الأمريكية لدى الأمم المتحدة وينعت كلامها بأوصاف غير.… https://t.co/KZsBj3Pq1N"', 'labels': ['ساخرة', 'جادة'], 'scores': [0.9320214986801147, 0.06797850877046585]}


In [5]:
import time
echantillon = val.sample(200, random_state=42)
t0 = time.time()
sortie = zs(echantillon.tweet.tolist(),
            candidate_labels=["ساخرة", "جادة"],
            hypothesis_template="هذه التغريدة {}.",
            batch_size=16)
print("temps 200 tweets :", round(time.time() - t0, 1), "s")
print(sortie[0])

temps 200 tweets : 1.3 s
{'sequence': '"مع كل يوم يمر وإخواننا في سجون #حماس المرتدة تتضاعف فاتورة الحساب التي ستدفعها الكلاب الحمساوية،،#وعد_صادق#حماس_تحتضر#حماس_تعتقل_السلفيين"', 'labels': ['ساخرة', 'جادة'], 'scores': [0.7312024831771851, 0.2687974274158478]}


In [6]:
import numpy as np

def score_sarcasme(textes, labels, template, batch_size=16):
    """Retourne P(sarcastique), normalisée entre les deux labels.
    labels[0] = étiquette sarcastique, labels[1] = étiquette sérieuse."""
    sorties = zs(list(textes), candidate_labels=labels,
                 hypothesis_template=template, batch_size=batch_size)
    scores = []
    for s in sorties:
        d = dict(zip(s["labels"], s["scores"]))
        scores.append(d[labels[0]])
    return np.array(scores)

In [7]:
from sklearn.metrics import average_precision_score, f1_score

y_va = val["sarcasm"].astype(int).values

formulations = {
    "arabe":   (["ساخرة", "جادة"], "هذه التغريدة {}."),
    "français": (["sarcastique", "sérieux"], "Ce tweet est {}."),
    "anglais": (["sarcastic", "serious"], "This tweet is {}."),
}

resultats_b1 = {}
for nom, (labels, template) in formulations.items():
    p = score_sarcasme(val.tweet.tolist(), labels, template)
    seuils = np.arange(0.05, 0.96, 0.01)
    f1s = [f1_score(y_va, p >= t, zero_division=0) for t in seuils]
    seuil = seuils[int(np.argmax(f1s))]
    resultats_b1[nom] = {"proba": p, "seuil": seuil}
    print(f"{nom:9s} | PR-AUC val {average_precision_score(y_va, p):.4f} | "
          f"seuil {seuil:.2f} | F1 val {max(f1s):.4f}")

arabe     | PR-AUC val 0.3375 | seuil 0.82 | F1 val 0.3871
français  | PR-AUC val 0.3495 | seuil 0.75 | F1 val 0.4119
anglais   | PR-AUC val 0.3407 | seuil 0.79 | F1 val 0.4040


In [8]:
meilleur = max(resultats_b1, key=lambda k: average_precision_score(y_va, resultats_b1[k]["proba"]))
print("Meilleure formulation (PR-AUC val) :", meilleur)
proba_b1 = resultats_b1[meilleur]["proba"]
seuil_b1 = resultats_b1[meilleur]["seuil"]

Meilleure formulation (PR-AUC val) : français


In [9]:
from transformers import pipeline

sa = pipeline("text-classification",
              model="CAMeL-Lab/bert-base-arabic-camelbert-da-sentiment",
              device=0, top_k=None)
print(sa.model.config.id2label)

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/305k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

{0: 'positive', 1: 'negative', 2: 'neutral'}


In [10]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

sorties_sa = sa(val.tweet.tolist(), batch_size=16, truncation=True)

# Label de sentiment prédit pour chaque tweet (le plus probable)
labels_sa = [max(d, key=lambda x: x["score"])["label"] for d in sorties_sa]

# Règle B2 : sentiment négatif => sarcastique
pred_b2 = np.array([lab == "negative" for lab in labels_sa]).astype(int)

print("part prédite sarcastique :", round(pred_b2.mean(), 4))
print("précision :", round(precision_score(y_va, pred_b2, zero_division=0), 4))
print("rappel    :", round(recall_score(y_va, pred_b2, zero_division=0), 4))
print("F1        :", round(f1_score(y_va, pred_b2, zero_division=0), 4))

part prédite sarcastique : 0.4056
précision : 0.332
rappel    : 0.7788
F1        : 0.4656


In [12]:
from src.results import ajouter_resultat, CHEMIN
import pandas as pd

ajouter_resultat({
    "modele": f"mDeBERTa NLI zero-shot ({meilleur})",
    "preprocessing": "dedup train",
    "graine": 42,
    "seuil": round(float(seuil_b1), 2),
    "pr_auc_val": round(float(average_precision_score(y_va, proba_b1)), 4),
    "f1_sarc_val": round(float(f1_score(y_va, proba_b1 >= seuil_b1)), 4),
    "note": "zero-shot, seuil sur val",
})

ajouter_resultat({
    "modele": "CAMeL sentiment, règle négatif=>sarcasme",
    "preprocessing": "dedup train",
    "graine": 42,
    "f1_sarc_val": round(float(f1_score(y_va, pred_b2, zero_division=0)), 4),
    "note": "contamination possible (ArSarcasm v1 construit sur ASTD/SemEval) ; non vérifiée",
})

print(pd.read_csv(CHEMIN).T)

                                                                       0  \
modele                 TF-IDF + LogisticRegression (class_weight=bala...   
preprocessing          dedup train + espaces normalisés; mots(1-2) + ...   
graine                                                                42   
seuil                                                               0.42   
pr_auc_val                                                        0.5217   
f1_sarc_val                                                       0.5153   
f1_sarc_val_seuil_0_5                                             0.5041   
accuracy_val_seuil                                                0.7721   
n_train                                                          11217.0   
n_val                                                             1255.0   
temps_entrainement_s                                                3.39   
params_entraines                                                186332.0   
versions    